### 🔥 Deep Learning Cave — Chapter 8: The Gemma 4 Sanctum

> *"Small is not weak. Small is precise."*

You've reached the deepest chamber yet. Here lies **Gemma 4 E2B** — Google DeepMind's 2.3B effective parameter multimodal model that punches orders of magnitude above its weight.

**Chapter 8** deconstructs the innovations that make Gemma 4's smallest model remarkable: **Per-Layer Embeddings**, **Alternating Local/Global Attention**, **Shared KV Cache**, **Dual RoPE**, and a **variable-resolution vision encoder**.

---

#### 🗺️ Your Position in the Cave

```
[■■■■■■■■■■] Chapter 1: PyTorch Foundations ✓
[■■■■■■■■■■] Chapter 2: The Original Transformer ✓
[■■■■■■■■■■] Chapter 3: Modern LLaMA ✓
[■■■■■■■■■■] Chapter 4: Vision Transformer ✓
[■■■■■■■■■■] Chapter 5: I-JEPA ✓
[■■■■■■■■■■] Chapter 6: Mixture of Experts ✓
[■■■■■■■■■■] Chapter 7: Knowledge Distillation ✓
[■■□□□□□□□□] Chapter 8: Gemma 4 E2B ← You are here
 ├── Per-Layer Embeddings (PLE)
 ├── Alternating Local + Global Attention
 ├── Shared KV Cache
 ├── Dual RoPE (standard + proportional)
 └── Variable-resolution vision encoding

[□□□□□□□□□□] Chapter 9: Coming soon...
```

**Blog**: [Gemma 4 on Hugging Face](https://huggingface.co/blog/gemma4)  
**Model**: `google/gemma-4-E2B-it` — 2.3B effective params, 128K context, text + image + audio

---

#### Gemma 4 E2B at a Glance

| Property | Value |
|----------|-------|
| Effective params | 2.3B |
| Total params (with embeddings) | 5.1B |
| Context window | 128K tokens |
| Local attention window | 512 tokens |
| Modalities | Text, Image, Video, Audio |
| MMLU Pro | 60.0% |
| MMMU Pro (Vision) | 44.2% |

```
┌──────────────────────────────────────────────────────────────────┐
│                    Gemma 4 E2B Architecture                      │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ┌─────────┐  ┌─────────┐  ┌─────────┐                          │
│  │  Image  │  │  Audio  │  │  Text   │  ← multimodal inputs     │
│  └────┬────┘  └────┬────┘  └────┬────┘                          │
│       │            │            │                                │
│  ┌────▼────┐  ┌────▼────┐       │                                │
│  │ SigLIP  │  │  USM    │       │                                │
│  │ Vision  │  │Conformer│       │                                │
│  │Encoder  │  │ Encoder │       │                                │
│  └────┬────┘  └────┬────┘       │                                │
│       │            │            │                                │
│       └────────────┴────────────┘                                │
│                    │                                             │
│                    ▼                                             │
│  ┌──────────────────────────────────┐                            │
│  │        Token Embeddings          │  ← main embeddings        │
│  │   + Per-Layer Embeddings (PLE)   │  ← NEW: layer-specific    │
│  └──────────────┬───────────────────┘                            │
│                 │                                                │
│    ┌────────────▼──────────────┐                                 │
│    │  Local Attention (512w)   │  ← even layers                 │
│    │  + PLE residual signal    │  ← per-layer conditioning      │
│    ├───────────────────────────┤                                 │
│    │  Global Full Attention    │  ← odd layers (every N)        │
│    │  + Shared KV Cache        │  ← reuse K,V from prev layer   │
│    ├───────────────────────────┤                                 │
│    │   ... × N layers ...      │                                 │
│    └────────────┬──────────────┘                                 │
│                 │                                                │
│                 ▼                                                │
│           ┌──────────┐                                           │
│           │  Output  │                                           │
│           └──────────┘                                           │
└──────────────────────────────────────────────────────────────────┘
```

---

#### Learning Path

| Part | Component | Key Concept |
|------|-----------|-------------|
| 1 | Setup & Config | E2B hyperparameters |
| 2 | Standard vs. Proportional RoPE | Dual RoPE for hybrid attention |
| 3 | Sliding Window Attention | Local context (512 tokens) |
| 4 | Global Full Attention | Long-range dependencies |
| 5 | Alternating Attention | Interleaving local + global layers |
| 6 | Per-Layer Embeddings (PLE) | The most novel Gemma 4 innovation |
| 7 | Shared KV Cache | Efficiency via K,V reuse |
| 8 | Vision Encoder | Variable token budget image encoding |
| 9 | Full E2B Model | Assembling all components |
| 10 | Inference Demo | Text generation walkthrough |

---

### Part 1: Setup and Configuration

🎯 **What it does**: Define Gemma 4 E2B architecture hyperparameters

🔧 **Why it matters**: E2B is a **Mixture of Depths** model — 2.3B *effective* parameters from a 5.1B total (with embeddings). The config captures the hybrid local/global attention pattern.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, Literal

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
@dataclass
class Gemma4E2BConfig:
    """Scaled-down config inspired by Gemma 4 E2B architecture.
    
    Real E2B: 2.3B eff. params, 18 layers, d_model=2048
    This demo:  ~40M params, 8 layers,  d_model=256
    Pattern is identical — just smaller for CPU-friendly training.
    """
    # Core dimensions
    vocab_size: int = 1000        # tiny vocab for demo (real: 262,144)
    d_model: int = 256            # hidden dim (real E2B: 2048)
    n_heads: int = 8              # query heads
    n_kv_heads: int = 4           # key/value heads — GQA
    d_ff: int = 1024              # FFN dim (real: uses SwiGLU, ~8192)
    n_layers: int = 8             # total decoder layers (real: 18)
    dropout: float = 0.0

    # Attention pattern (Gemma 4's hybrid design)
    # Even-indexed layers → local sliding window
    # Odd-indexed layers  → global full attention
    local_window_size: int = 16   # real E2B: 512 tokens
    global_attention_every_n: int = 2  # every 2nd layer is global

    # RoPE configurations
    rope_theta_local: float = 10000.0    # standard RoPE for local layers
    rope_theta_global: float = 1_000_000.0  # proportional RoPE for global

    # Per-Layer Embeddings (PLE) — Gemma 4's key innovation
    use_ple: bool = True
    ple_dim: int = 64             # much smaller than d_model (real: ~256 vs 2048)

    # Shared KV Cache
    num_kv_shared_layers: int = 2  # last N layers share KV from earlier

    # Vision config (simplified)
    image_token_budgets: list = field(default_factory=lambda: [70, 140, 280])
    vision_patch_size: int = 14
    vision_d_model: int = 128

    # Context
    max_seq_len: int = 256        # real E2B: 128K

    @property
    def head_dim(self):
        return self.d_model // self.n_heads

    @property
    def is_local_layer(self):
        """Returns a list: True = local, False = global, for each layer."""
        return [i % self.global_attention_every_n != 0 for i in range(self.n_layers)]


config = Gemma4E2BConfig()

print("=" * 60)
print("  Gemma 4 E2B Config (Demo — scaled for CPU)")
print("=" * 60)
print(f"  d_model:            {config.d_model}  (real E2B: 2048)")
print(f"  n_layers:           {config.n_layers}  (real E2B: 18)")
print(f"  n_heads (Q):        {config.n_heads}  (GQA query heads)")
print(f"  n_kv_heads:         {config.n_kv_heads}  (GQA key/value heads)")
print(f"  local_window:       {config.local_window_size}  (real E2B: 512)")
print(f"  PLE dim:            {config.ple_dim}  (real E2B: ~256)")
print(f"  Shared KV layers:   last {config.num_kv_shared_layers}")
print(f"  RoPE θ (local):     {config.rope_theta_local:,.0f}")
print(f"  RoPE θ (global):    {config.rope_theta_global:,.0f}")
print()
print("  Layer attention pattern:")
for i, is_local in enumerate(config.is_local_layer):
    attn_type = "LOCAL  (window={})".format(config.local_window_size) if is_local else "GLOBAL (full context)"
    shared = " ← SHARED KV" if i >= config.n_layers - config.num_kv_shared_layers else ""
    print(f"    Layer {i}: {attn_type}{shared}")
print("=" * 60)

---

### Part 2: Dual RoPE — Standard vs. Proportional

🎯 **What it does**: Apply different RoPE configurations for local vs. global attention layers

🔧 **Why it matters**: Local attention sees only 512 tokens — standard RoPE is fine. Global attention spans 128K tokens — standard RoPE's frequency range isn't designed for that. **Proportional RoPE** (θ=1M) stretches the positional encoding to handle extreme context lengths gracefully.

In [ ]:
def precompute_rope_freqs(head_dim: int, max_seq_len: int, theta: float) -> torch.Tensor:
    """Precompute RoPE rotation frequencies.
    
    RoPE encodes position by rotating query/key vectors in 2D subspaces.
    Each dimension pair (2i, 2i+1) gets its own rotation frequency.
    
    Standard RoPE (θ=10,000):  Good for short sequences
    Proportional RoPE (θ=1M):  Good for very long sequences
    
    Higher θ → lower base frequencies → slower rotation → better long-range
    """
    # Frequencies: θ_i = 1 / theta^(2i/d) for i in [0, head_dim/2)
    i = torch.arange(0, head_dim, 2).float()
    freqs = 1.0 / (theta ** (i / head_dim))   # (head_dim/2,)
    
    # Position indices
    positions = torch.arange(max_seq_len).float()  # (seq_len,)
    
    # Outer product: each position × each frequency
    angles = torch.outer(positions, freqs)  # (seq_len, head_dim/2)
    
    # Build complex exponentials: e^(i * angle) = cos + i*sin
    freqs_cis = torch.polar(torch.ones_like(angles), angles)  # complex
    return freqs_cis  # (seq_len, head_dim/2)


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """Apply rotary position embeddings to query or key tensor.
    
    Rotation in pairs: [x0, x1] → [x0*cos - x1*sin, x0*sin + x1*cos]
    Equivalent to multiplying complex numbers.
    """
    # x: (B, n_heads, seq_len, head_dim)
    B, H, S, D = x.shape
    
    # View as complex: pairs of floats → complex numbers
    x_complex = torch.view_as_complex(x.float().reshape(B, H, S, D // 2, 2))
    
    # freqs_cis: (seq_len, head_dim/2) → broadcast over B, H
    freqs = freqs_cis[:S].unsqueeze(0).unsqueeze(0)  # (1, 1, S, D/2)
    
    # Complex multiplication = rotation
    x_rotated = x_complex * freqs
    
    # Back to real
    return torch.view_as_real(x_rotated).reshape(B, H, S, D).type_as(x)


# Precompute for both attention types
rope_local  = precompute_rope_freqs(config.head_dim, config.max_seq_len, config.rope_theta_local)
rope_global = precompute_rope_freqs(config.head_dim, config.max_seq_len, config.rope_theta_global)

print(f"RoPE freqs shape: {rope_local.shape}  (seq_len, head_dim/2)")
print()

# Visualize: how fast do rotation angles change with position?
positions = torch.arange(config.max_seq_len).float()
dim_idx = torch.arange(0, config.head_dim, 2).float()

freqs_local  = 1.0 / (config.rope_theta_local  ** (dim_idx / config.head_dim))
freqs_global = 1.0 / (config.rope_theta_global ** (dim_idx / config.head_dim))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rotation angle for first few dimension pairs
for d in [0, 2, 4, 6]:
    angles_local  = positions.numpy() * freqs_local[d//2].item()
    angles_global = positions.numpy() * freqs_global[d//2].item()
    axes[0].plot(positions.numpy(), np.sin(angles_local),  label=f'dim {d}', linewidth=1.5)
    axes[1].plot(positions.numpy(), np.sin(angles_global), label=f'dim {d}', linewidth=1.5)

axes[0].set_title(f'Standard RoPE (θ={config.rope_theta_local:.0f})\nLocal Attention Layers', fontweight='bold')
axes[1].set_title(f'Proportional RoPE (θ={config.rope_theta_global:.0f})\nGlobal Attention Layers', fontweight='bold')
for ax in axes:
    ax.set_xlabel('Position')
    ax.set_ylabel('sin(angle) — rotation signal')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].text(0.5, 0.05, 'Fast rotations → good for local (512 token) context', 
             ha='center', transform=axes[0].transAxes,
             bbox=dict(boxstyle='round', facecolor='#d6eaf8', alpha=0.9), fontsize=9)
axes[1].text(0.5, 0.05, 'Slow rotations → good for global (128K token) context',
             ha='center', transform=axes[1].transAxes,
             bbox=dict(boxstyle='round', facecolor='#d5f5e3', alpha=0.9), fontsize=9)

plt.suptitle('Dual RoPE: Gemma 4 uses different θ for local vs. global layers', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

💡 **Key Insight**: Standard RoPE (θ=10K) has frequencies that complete multiple full rotations over just 512 positions — perfect for local attention windows. Proportional RoPE (θ=1M) barely completes a single rotation over 128K positions — its slow drift carries meaningful positional signal at extreme distances without aliasing.

Using the **same θ for both** would either make local attention positionally imprecise OR make global attention unable to distinguish positions 50K vs 51K apart.

---

### Part 3: Sliding Window (Local) Attention

🎯 **What it does**: Restrict each token's attention to a 512-token window

🔧 **Why it matters**: Full attention is O(n²) in sequence length. At 128K tokens that's catastrophic. Sliding window attention is O(n·w) — linear in sequence length. Most useful context is local anyway.

In [ ]:
def make_sliding_window_mask(seq_len: int, window_size: int) -> torch.Tensor:
    """Create a causal sliding window attention mask.
    
    Each position i can only attend to positions in [i-window_size, i].
    (Plus causal: no attending to future positions.)
    
    Returns: (seq_len, seq_len) boolean mask where True = MASKED (blocked)
    """
    # Causal mask: position i cannot attend to j > i
    causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
    
    # Window mask: position i cannot attend to j < i - window_size
    row_idx = torch.arange(seq_len).unsqueeze(1)  # (seq_len, 1)
    col_idx = torch.arange(seq_len).unsqueeze(0)  # (1, seq_len)
    window_mask = (row_idx - col_idx) > window_size
    
    return causal_mask | window_mask  # True = blocked


def make_causal_mask(seq_len: int) -> torch.Tensor:
    """Standard causal mask for full global attention."""
    return torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)


# Visualize both masks
demo_len = 32
demo_window = 8

local_mask  = make_sliding_window_mask(demo_len, demo_window)
global_mask = make_causal_mask(demo_len)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Local mask
im0 = axes[0].imshow(~local_mask.numpy(), cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[0].set_title(f'Local Attention (window={demo_window})\nO(n·w) complexity', fontweight='bold')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
axes[0].text(0.5, -0.15, '← Attends only to recent tokens', ha='center',
             transform=axes[0].transAxes, fontsize=9)

# Global mask
im1 = axes[1].imshow(~global_mask.numpy(), cmap='Oranges', aspect='auto', vmin=0, vmax=1)
axes[1].set_title(f'Global Attention (full causal)\nO(n²) complexity', fontweight='bold')
axes[1].set_xlabel('Key position')
axes[1].text(0.5, -0.15, '← Attends to all previous tokens', ha='center',
             transform=axes[1].transAxes, fontsize=9)

# Complexity comparison
seq_lens = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
w = 512  # real E2B window
local_ops  = [n * w for n in seq_lens]
global_ops = [n * n for n in seq_lens]

axes[2].plot([s/1000 for s in seq_lens], [o/1e9 for o in global_ops], 
             'r-o', linewidth=2.5, markersize=6, label='Full attention O(n²)')
axes[2].plot([s/1000 for s in seq_lens], [o/1e9 for o in local_ops],
             'b-s', linewidth=2.5, markersize=6, label=f'Sliding window O(n·{w})')
axes[2].set_xlabel('Sequence length (K tokens)')
axes[2].set_ylabel('Attention ops (Billions)')
axes[2].set_title('Computational Complexity', fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)
axes[2].set_yscale('log')

plt.suptitle('Local (Sliding Window) vs. Global (Full) Attention', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("At 128K tokens:")
n = 128_000
w = 512
print(f"  Full attention:   {n*n/1e9:.1f}B operations")
print(f"  Sliding window:   {n*w/1e9:.3f}B operations")
print(f"  Speedup:          {(n*n)/(n*w):.0f}×")

💡 **Key Insight**: At 128K tokens, full attention requires **16 trillion operations** per layer. Sliding window (w=512) needs only **65 billion** — a 250× reduction. The trick: most information a token needs for generation is local. The long-range context is captured by the interleaved **global layers**.

---

### Part 4: Grouped Query Attention (GQA)

🎯 **What it does**: Fewer KV heads than Q heads — reducing KV cache size

🔧 **Why it matters**: Gemma 4 E2B uses GQA with n_kv_heads < n_heads. The KV cache grows with sequence length — at 128K context, shrinking KV heads from 8→4 halves the memory footprint.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Grouped Query Attention — as used in Gemma 4 E2B.
    
    n_heads (Q) > n_kv_heads (K, V)
    Each KV head is shared across (n_heads / n_kv_heads) query heads.
    
    MHA:  n_heads Q, n_heads K, n_heads V  (original Transformer)
    GQA:  n_heads Q, n_kv_heads K, n_kv_heads V  (Gemma, LLaMA-3, etc.)
    MQA:  n_heads Q, 1 K, 1 V  (extreme case)
    """

    def __init__(self, config: Gemma4E2BConfig, is_local: bool = True, layer_idx: int = 0):
        super().__init__()
        self.n_heads    = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.head_dim   = config.head_dim
        self.d_model    = config.d_model
        self.is_local   = is_local
        self.window     = config.local_window_size
        self.groups     = config.n_heads // config.n_kv_heads  # Q heads per KV head

        # Projections
        self.wq  = nn.Linear(config.d_model, config.n_heads    * config.head_dim, bias=False)
        self.wk  = nn.Linear(config.d_model, config.n_kv_heads * config.head_dim, bias=False)
        self.wv  = nn.Linear(config.d_model, config.n_kv_heads * config.head_dim, bias=False)
        self.wo  = nn.Linear(config.n_heads * config.head_dim, config.d_model,    bias=False)

        self.scale = self.head_dim ** -0.5

    def forward(
        self,
        x: torch.Tensor,
        rope_freqs: torch.Tensor,
        cached_kv: Optional[tuple] = None,   # for shared KV cache
    ) -> tuple[torch.Tensor, tuple]:
        B, S, _ = x.shape

        # Project to Q, K, V
        q = self.wq(x).view(B, S, self.n_heads,    self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, S, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, S, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to Q and K
        q = apply_rope(q, rope_freqs)
        k = apply_rope(k, rope_freqs)

        # Override K, V with shared KV cache if provided
        if cached_kv is not None:
            k, v = cached_kv  # reuse K, V from earlier layer!

        # Expand KV heads to match Q heads (GQA repeat)
        k_expanded = k.repeat_interleave(self.groups, dim=1)  # (B, n_heads, S, head_dim)
        v_expanded = v.repeat_interleave(self.groups, dim=1)

        # Attention scores
        scores = torch.matmul(q, k_expanded.transpose(-2, -1)) * self.scale

        # Apply mask
        if self.is_local:
            mask = make_sliding_window_mask(S, self.window).to(x.device)
        else:
            mask = make_causal_mask(S).to(x.device)

        scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        weights = F.softmax(scores, dim=-1)

        # Weighted sum
        out = torch.matmul(weights, v_expanded)           # (B, n_heads, S, head_dim)
        out = out.transpose(1, 2).reshape(B, S, -1)       # (B, S, d_model)
        out = self.wo(out)

        return out, (k, v)  # return K, V for potential sharing


# Quick test
attn = GroupedQueryAttention(config, is_local=True, layer_idx=0).to(device)
x_test = torch.randn(2, 32, config.d_model).to(device)
out_test, (k_cache, v_cache) = attn(x_test, rope_local.to(device))

print("GQA test:")
print(f"  Input:  {x_test.shape}")
print(f"  Output: {out_test.shape}")
print(f"  K cache:{k_cache.shape}  (n_kv_heads={config.n_kv_heads}, not {config.n_heads})")
print(f"  V cache:{v_cache.shape}")

q_params  = config.n_heads    * config.head_dim * config.d_model
kv_params = config.n_kv_heads * config.head_dim * config.d_model * 2
print(f"\n  Q projection params: {q_params:,}")
print(f"  KV projection params: {kv_params:,}  ({config.n_heads//config.n_kv_heads}× smaller than MHA)")

---

### Part 5: Alternating Local + Global Attention

🎯 **What it does**: Interleave local (sliding window) and global (full causal) attention layers

🔧 **Why it matters**: Local-only would lose long-range context. Global-only is O(n²) at 128K. The alternating pattern gives you **both** — local layers handle most computation efficiently, global layers periodically integrate the full context.

In [ ]:
# Visualize the alternating pattern
fig, ax = plt.subplots(figsize=(12, 4))

n_layers = config.n_layers
layer_types = config.is_local_layer
shared_start = n_layers - config.num_kv_shared_layers

for i, is_local in enumerate(layer_types):
    color   = '#3498db' if is_local else '#e67e22'
    label   = 'LOCAL' if is_local else 'GLOBAL'
    shared  = i >= shared_start
    edge    = '#c0392b' if shared else '#2c3e50'
    lw      = 3 if shared else 1.5

    rect = mpatches.FancyBboxPatch(
        (i * 1.2, 0.1), 1.0, 0.8,
        boxstyle='round,pad=0.05',
        facecolor=color, edgecolor=edge, linewidth=lw, alpha=0.85
    )
    ax.add_patch(rect)
    ax.text(i * 1.2 + 0.5, 0.5, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')
    ax.text(i * 1.2 + 0.5, -0.05, str(i), ha='center', va='top', fontsize=8)
    
    if shared:
        ax.text(i * 1.2 + 0.5, 1.05, '↑ Shared\n  KV', ha='center', va='bottom',
                fontsize=7, color='#c0392b', fontweight='bold')

ax.set_xlim(-0.2, n_layers * 1.2)
ax.set_ylim(-0.3, 1.4)
ax.axis('off')

blue_patch  = mpatches.Patch(color='#3498db', label='Local (sliding window, 512 tok)')
orange_patch = mpatches.Patch(color='#e67e22', label='Global (full causal, 128K tok)')
red_line    = mpatches.Patch(color='#fadbd8', edgecolor='#c0392b', linewidth=2, label='Shared KV Cache layers')
ax.legend(handles=[blue_patch, orange_patch, red_line], loc='lower center', fontsize=9, ncol=3)

ax.set_title('Gemma 4 E2B: Alternating Attention Pattern (8 layers, simplified)',
             fontsize=12, fontweight='bold')
ax.text(0.5, -0.25, 'Layer index →', ha='center', transform=ax.transAxes, fontsize=9)
plt.tight_layout()
plt.show()

# Real E2B pattern (18 layers)
print("Real Gemma 4 E2B (18 layers):")
real_pattern = ['LOCAL' if i % 2 != 0 else 'GLOBAL' for i in range(18)]
print('  ' + ' | '.join(f'{i}:{p[:3]}' for i, p in enumerate(real_pattern)))
n_local  = sum(1 for p in real_pattern if p == 'LOCAL')
n_global = sum(1 for p in real_pattern if p == 'GLOBAL')
print(f"\n  Local layers:  {n_local}/18  ({100*n_local/18:.0f}% of compute at O(n·512))")
print(f"  Global layers: {n_global}/18  ({100*n_global/18:.0f}% of compute at O(n²))")

---

### Part 6: Per-Layer Embeddings (PLE) — Gemma 4's Most Novel Innovation

🎯 **What it does**: Add a second, smaller embedding table that injects a **per-layer, per-token conditioning signal** into every decoder layer

🔧 **Why it matters**: Standard transformers use one embedding lookup — the same vector flows through all layers unchanged (modulo residual stream). PLE gives each layer its **own view** of each token's identity, enabling layer-specific specialization with minimal parameter cost.

In [ ]:
class PerLayerEmbedding(nn.Module):
    """Per-Layer Embeddings (PLE) — Gemma 4's signature innovation.
    
    Standard transformer:
        token_id → embedding(token_id) → [same vector used by all layers]
    
    Gemma 4 with PLE:
        token_id → main_embedding(token_id)  → flows through all layers normally
                 → ple_embedding(token_id)   → ALSO injected into EVERY layer
    
    For each decoder layer, PLE provides:
        - A token-specific residual signal (ple_dim << d_model)
        - Projected back up to d_model before adding
        - Injected after attention AND after FFN (via lightweight residual block)
    
    For multimodal tokens (images, audio):
        - PLE uses the pad_token_id → neutral signal (no spurious conditioning)
    """

    def __init__(self, config: Gemma4E2BConfig):
        super().__init__()
        # The small PLE embedding table
        self.ple_embed = nn.Embedding(config.vocab_size, config.ple_dim)

        # Per-layer projection: ple_dim → d_model (one per layer)
        self.layer_projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(config.ple_dim, config.d_model, bias=False),
                nn.ReLU(),
                nn.Linear(config.d_model, config.d_model, bias=False),
            )
            for _ in range(config.n_layers)
        ])

    def get_signals(self, token_ids: torch.Tensor) -> list[torch.Tensor]:
        """Compute PLE signals for all layers.
        
        Args:
            token_ids: (B, S) token indices
        Returns:
            List of (B, S, d_model) tensors, one per layer
        """
        # Base PLE vector: (B, S, ple_dim)
        ple_base = self.ple_embed(token_ids)

        # Project to d_model for each layer independently
        return [proj(ple_base) for proj in self.layer_projections]


# Demonstrate PLE
ple = PerLayerEmbedding(config).to(device)
token_ids = torch.randint(0, config.vocab_size, (2, 32)).to(device)

signals = ple.get_signals(token_ids)

print("Per-Layer Embedding demo:")
print(f"  Token IDs shape:    {token_ids.shape}")
print(f"  PLE base shape:     {ple.ple_embed(token_ids).shape}  (ple_dim={config.ple_dim})")
print(f"  PLE signals:        {len(signals)} tensors (one per layer)")
print(f"  Each signal shape:  {signals[0].shape}")

ple_params  = config.vocab_size * config.ple_dim
main_params = config.vocab_size * config.d_model
proj_params = config.n_layers * (config.ple_dim * config.d_model + config.d_model * config.d_model)
print(f"\n  Parameter count:")
print(f"    Main embedding:    {main_params:>10,}")
print(f"    PLE embedding:     {ple_params:>10,}  ({100*ple_params/main_params:.1f}% of main)")
print(f"    PLE projections:   {proj_params:>10,}")
print(f"    Total PLE cost:    {ple_params+proj_params:>10,}  (modest overhead)")

# Visualize: how different are PLE signals across layers for the same token?
with torch.no_grad():
    same_token = torch.tensor([[42]]).to(device)  # single token
    sigs = ple.get_signals(same_token)
    sig_matrix = torch.stack([s[0, 0] for s in sigs]).cpu().numpy()  # (n_layers, d_model)

plt.figure(figsize=(10, 4))
plt.imshow(sig_matrix, cmap='RdBu', aspect='auto')
plt.colorbar(label='PLE signal value')
plt.xlabel('d_model dimension')
plt.ylabel('Layer index')
plt.title('PLE Signals for Token #42 Across All Layers\nEach row = layer-specific conditioning', 
          fontweight='bold')
plt.tight_layout()
plt.show()
print("\n↑ Different rows = different layer-specific conditioning for the SAME token")
print("  This lets each layer specialize its understanding of each token's role.")

💡 **Key Insight**: In a standard transformer, every layer sees the same token representation (filtered only through the residual stream). PLE gives each layer a **direct, unfiltered signal** about the token's identity — like a sidebar note to each layer saying "hey, this token is token #42, here's what that means for you specifically."

This is especially valuable for **multimodal models**: image tokens use the pad token ID in PLE → neutral signal, preventing any spurious text-token conditioning from leaking into image-derived positions.

Parameter cost: `vocab_size × ple_dim` ≈ `262,144 × 256` ≈ 67M — cheap relative to the 2.3B model.

---

### Part 7: Shared KV Cache

🎯 **What it does**: The last N layers skip computing their own K and V — they reuse K,V from the last non-shared layer of the same attention type

🔧 **Why it matters**: At 128K context, the KV cache dominates memory. Reusing K,V in the final layers reduces both the KV cache footprint AND the K/V projection compute, with minimal quality loss.

In [ ]:
# Demonstrate and visualize the Shared KV Cache pattern

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

n = config.n_layers
shared_start = n - config.num_kv_shared_layers

# Left: show which layers compute vs reuse KV
ax = axes[0]
ax.set_xlim(0, 4)
ax.set_ylim(-0.5, n + 0.5)
ax.axis('off')
ax.set_title('Shared KV Cache: Which Layers Compute K,V?', fontweight='bold', fontsize=11)

for i in range(n):
    y = n - 1 - i
    is_shared = (i >= shared_start)
    is_local  = config.is_local_layer[i]

    # Layer box
    color = '#fadbd8' if is_shared else '#d5f5e3'
    rect = mpatches.FancyBboxPatch((0.1, y - 0.35), 1.5, 0.7,
                                    boxstyle='round,pad=0.05',
                                    facecolor=color, edgecolor='#2c3e50', linewidth=1.5)
    ax.add_patch(rect)
    label = f"L{i}: {'LOCAL' if is_local else 'GLOBAL'}"
    ax.text(0.85, y, label, ha='center', va='center', fontsize=9, fontweight='bold')

    # KV status
    if is_shared:
        ax.text(2.0, y, '⟵ reuses K,V\n   (no wk, wv)', ha='left', va='center', 
                fontsize=8, color='#c0392b')
    else:
        ax.text(2.0, y, '✓ computes K,V', ha='left', va='center', 
                fontsize=8, color='#27ae60')

    # Arrow from non-shared to shared
    if i == shared_start - 1:
        ax.annotate('', xy=(0.85, n - 1 - (i + 1) + 0.3),
                    xytext=(0.85, y - 0.35),
                    arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2,
                                    connectionstyle='arc3,rad=0.3'))
        ax.text(0.1, n - 1 - i - 0.5, 'K,V flows down →', fontsize=8, color='#e74c3c', style='italic')

# Right: memory savings
ax2 = axes[1]
seq_lens = [4096, 8192, 16384, 32768, 65536, 131072]
d_kv = config.n_kv_heads * config.head_dim  # per-token KV size

# Without shared KV: n_layers × 2 × d_kv × seq_len
# With shared KV: (n_layers - n_shared) + n_shared/n_attn_types × 2 × d_kv × seq_len
kv_without = [n * 2 * d_kv * s * 2 / 1e6 for s in seq_lens]  # fp16, MB
kv_with    = [(n - config.num_kv_shared_layers) * 2 * d_kv * s * 2 / 1e6 for s in seq_lens]

ax2.plot([s/1000 for s in seq_lens], kv_without, 'r-o', linewidth=2.5, markersize=6,
         label='Standard KV cache (all layers)')
ax2.plot([s/1000 for s in seq_lens], kv_with,    'b-s', linewidth=2.5, markersize=6,
         label=f'Shared KV cache (last {config.num_kv_shared_layers} layers reuse)')
ax2.fill_between([s/1000 for s in seq_lens], kv_with, kv_without, alpha=0.2, color='green',
                 label='Memory saved')
ax2.set_xlabel('Sequence length (K tokens)')
ax2.set_ylabel('KV Cache size (MB, fp16)')
ax2.set_title(f'KV Cache Memory: {config.num_kv_shared_layers}/{n} Layers Shared', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Shared KV Cache: Reduce Memory Without Sacrificing Quality', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

savings = (kv_without[-1] - kv_with[-1]) / kv_without[-1] * 100
print(f"At 128K tokens:")
print(f"  Standard KV cache: {kv_without[-1]:.1f} MB")
print(f"  Shared KV cache:   {kv_with[-1]:.1f} MB")
print(f"  Memory saved:      {savings:.1f}%  ({config.num_kv_shared_layers} of {n} layers)")

💡 **Key Insight**: The Shared KV Cache is a form of **parameter and compute tying** — the assumption is that the last few layers' K,V projections are redundant enough to reuse those from the previous layer. Empirically, Google found this has **minimal quality impact** while providing meaningful savings for long-context inference and on-device deployment.

---

### Part 8: Vision Encoder — Variable Token Budget

🎯 **What it does**: Encode images to a configurable number of tokens (70 / 140 / 280 / 560 / 1120)

🔧 **Why it matters**: More tokens = richer image understanding but higher compute cost. The variable budget lets users choose the speed/quality tradeoff per query. A UI screenshot needs more tokens than a simple icon.

In [ ]:
class VisionPatchEmbedding(nn.Module):
    """Encode image patches with 2D learned positional embeddings.

    Gemma 4's vision encoder uses:
    - SigLIP-based ViT backbone
    - Learned 2D positional embeddings (not sinusoidal)
    - Multidimensional RoPE in the encoder
    - Aspect ratio preservation
    - Variable token budget output (70/140/280/560/1120 tokens)

    This is a simplified version: patch embed → linear pool → token budget.
    """

    def __init__(self, config: Gemma4E2BConfig):
        super().__init__()
        p = config.vision_patch_size  # 14
        d = config.vision_d_model

        # Patch projection (equivalent to ViT patch embedding)
        self.patch_proj = nn.Conv2d(
            in_channels=3, out_channels=d,
            kernel_size=p, stride=p
        )

        # Learned 2D positional embedding (not sinusoidal)
        # For a 224×224 image with patch_size=14: 16×16 = 256 patches
        self.max_patches_h = 16
        self.max_patches_w = 16
        self.pos_embed_h = nn.Embedding(self.max_patches_h, d // 2)
        self.pos_embed_w = nn.Embedding(self.max_patches_w, d // 2)

        # Linear projection to LLM d_model
        self.output_proj = nn.Linear(d, config.d_model)

    def forward(self, images: torch.Tensor, token_budget: int = 140) -> torch.Tensor:
        """Encode images to exactly `token_budget` tokens.

        Args:
            images: (B, 3, H, W) image batch
            token_budget: target number of visual tokens (70/140/280/560/1120)
        Returns:
            (B, token_budget, d_model) visual token sequence
        """
        B, C, H, W = images.shape

        # 1. Extract patches via conv
        patches = self.patch_proj(images)     # (B, d, ph, pw)
        B, D, ph, pw = patches.shape

        # 2. Add 2D learned positional embeddings
        h_idx = torch.arange(ph, device=images.device)
        w_idx = torch.arange(pw, device=images.device)
        # Clamp to max_patches (for safety)
        h_idx = h_idx.clamp(0, self.max_patches_h - 1)
        w_idx = w_idx.clamp(0, self.max_patches_w - 1)

        pos_h = self.pos_embed_h(h_idx)  # (ph, d/2)
        pos_w = self.pos_embed_w(w_idx)  # (pw, d/2)

        # Broadcast and concatenate to make (ph, pw, d)
        pos_h_grid = pos_h.unsqueeze(1).expand(-1, pw, -1)  # (ph, pw, d/2)
        pos_w_grid = pos_w.unsqueeze(0).expand(ph, -1, -1)  # (ph, pw, d/2)
        pos_2d = torch.cat([pos_h_grid, pos_w_grid], dim=-1) # (ph, pw, d)
        pos_2d = pos_2d.permute(2, 0, 1).unsqueeze(0)        # (1, d, ph, pw)

        patches = patches + pos_2d  # add 2D position info

        # 3. Flatten patches to sequence
        n_patches = ph * pw
        patches_flat = patches.reshape(B, D, n_patches).transpose(1, 2)  # (B, n_patches, D)

        # 4. Adaptive pool to target token budget
        # In real Gemma 4: uses more sophisticated pooling with aspect-ratio preservation
        # Here: simple adaptive avg pooling over the sequence dimension
        patches_pool = F.adaptive_avg_pool1d(
            patches_flat.transpose(1, 2),  # (B, D, n_patches)
            token_budget
        ).transpose(1, 2)  # (B, token_budget, D)

        # 5. Project to LLM dimension
        return self.output_proj(patches_pool)  # (B, token_budget, d_model)


# Demo: different token budgets
vision_enc = VisionPatchEmbedding(config).to(device)
dummy_image = torch.randn(1, 3, 224, 224).to(device)

budgets = config.image_token_budgets
print("Variable Token Budget demo (224×224 image):")
print(f"  Raw patches: {(224//config.vision_patch_size)**2} patches")
print()

results = {}
for budget in budgets:
    out = vision_enc(dummy_image, token_budget=budget)
    results[budget] = out
    compression = (224 // config.vision_patch_size)**2 / budget
    print(f"  Budget={budget:4d} tokens: {out.shape}  "
          f"(compression {compression:.1f}× from raw)")

# Visualize token budget tradeoff
real_budgets = [70, 140, 280, 560, 1120]
context_cost = [b / 128_000 * 100 for b in real_budgets]  # % of 128K context

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(real_budgets)), real_budgets, 
              color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'],
              alpha=0.85, edgecolor='#2c3e50', linewidth=1.5)
ax.set_xticks(range(len(real_budgets)))
ax.set_xticklabels([f'{b}\ntokens\n({c:.1f}% ctx)' for b, c in zip(real_budgets, context_cost)])
ax.set_ylabel('Visual tokens consumed')
ax.set_title('Gemma 4 E2B: Configurable Image Token Budget\n(Trade-off: richer → more context used)',
             fontweight='bold')

for bar, budget in zip(bars, real_budgets):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{budget}', ha='center', fontweight='bold', fontsize=10)

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

💡 **Key Insight**: Gemma 4's vision encoder uses **learned 2D positional embeddings** (not sinusoidal) — crucial for spatial understanding of images. The row and column embeddings are learned independently and concatenated, giving the model fine-grained 2D spatial awareness.

The variable token budget is a practical design choice: 70 tokens for thumbnails/icons, 1120 tokens for dense document analysis. Users can dial this based on their latency/quality needs. At 128K context, even 1120 visual tokens leave 126K for text — plenty for long conversations.

---

### Part 9: Full Gemma 4 E2B Decoder (Assembled)

🎯 **What it does**: Assemble all components into a working E2B-style decoder

🔧 **Why it matters**: Seeing how PLE, alternating attention, shared KV, and GQA all interact in a single forward pass.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization — used in Gemma 4 instead of LayerNorm."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


class SwiGLU(nn.Module):
    """SwiGLU FFN — used in Gemma 4 (Swish-gated linear unit).
    
    Outperforms standard FFN (ReLU/GELU) on most language tasks.
    Output = Swish(xW1) ⊙ (xW2) → projected by W3
    """
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.w2 = nn.Linear(d_model, d_ff, bias=False)  # value
        self.w3 = nn.Linear(d_ff, d_model, bias=False)  # output

    def forward(self, x):
        gate   = F.silu(self.w1(x))   # SiLU ≈ Swish
        value  = self.w2(x)
        return self.w3(gate * value)


class Gemma4DecoderLayer(nn.Module):
    """Single Gemma 4 decoder layer.

    Structure:
        Pre-norm RMSNorm → Attention (local or global, GQA) → residual
        + PLE signal injection                                
        Pre-norm RMSNorm → SwiGLU FFN → residual              
        + PLE signal injection                                
    """

    def __init__(self, config: Gemma4E2BConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        is_local = config.is_local_layer[layer_idx]

        self.norm1 = RMSNorm(config.d_model)
        self.attn  = GroupedQueryAttention(config, is_local=is_local, layer_idx=layer_idx)
        self.norm2 = RMSNorm(config.d_model)
        self.ffn   = SwiGLU(config.d_model, config.d_ff)

        # PLE injection gates (learnable scale for each injection point)
        self.ple_gate_attn = nn.Parameter(torch.ones(1) * 0.1)  # start small
        self.ple_gate_ffn  = nn.Parameter(torch.ones(1) * 0.1)

    def forward(
        self,
        x: torch.Tensor,
        rope_freqs: torch.Tensor,
        ple_signal: torch.Tensor,    # (B, S, d_model) — from PLE for this layer
        cached_kv: Optional[tuple] = None,
    ) -> tuple[torch.Tensor, tuple]:
        # 1. Pre-norm → Attention → residual + PLE
        attn_out, kv = self.attn(self.norm1(x), rope_freqs, cached_kv=cached_kv)
        x = x + attn_out + self.ple_gate_attn * ple_signal  # PLE injection!

        # 2. Pre-norm → FFN → residual + PLE
        x = x + self.ffn(self.norm2(x)) + self.ple_gate_ffn * ple_signal  # PLE injection!

        return x, kv


class Gemma4E2B(nn.Module):
    """Full Gemma 4 E2B-style decoder model.

    Combines:
    1. Token embeddings + Per-Layer Embeddings (PLE)
    2. Alternating local/global GQA layers
    3. Shared KV Cache (last num_kv_shared_layers reuse K,V)
    4. Dual RoPE (standard for local, proportional for global)
    5. RMSNorm + SwiGLU
    """

    def __init__(self, config: Gemma4E2BConfig):
        super().__init__()
        self.config = config

        # Token embeddings (main)
        self.token_embed = nn.Embedding(config.vocab_size, config.d_model)

        # Per-Layer Embeddings
        self.ple = PerLayerEmbedding(config) if config.use_ple else None

        # Precompute both RoPE variants
        rope_l = precompute_rope_freqs(config.head_dim, config.max_seq_len, config.rope_theta_local)
        rope_g = precompute_rope_freqs(config.head_dim, config.max_seq_len, config.rope_theta_global)
        self.register_buffer('rope_local',  rope_l)
        self.register_buffer('rope_global', rope_g)

        # Decoder layers
        self.layers = nn.ModuleList([
            Gemma4DecoderLayer(config, i) for i in range(config.n_layers)
        ])

        # Final RMSNorm + output head
        self.norm = RMSNorm(config.d_model)
        self.head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying: output head shares weights with token_embed
        self.head.weight = self.token_embed.weight

    def forward(
        self,
        token_ids: torch.Tensor,       # (B, S)
        image_tokens: Optional[torch.Tensor] = None,  # (B, n_img, d_model)
    ) -> torch.Tensor:
        B, S = token_ids.shape

        # 1. Token embeddings
        x = self.token_embed(token_ids)  # (B, S, d_model)

        # 2. Prepend image tokens if present (multimodal)
        if image_tokens is not None:
            x = torch.cat([image_tokens, x], dim=1)  # (B, n_img+S, d_model)
            # PLE for image positions uses pad_id (neutral)
            pad_ids = torch.zeros(B, image_tokens.shape[1], dtype=torch.long, device=token_ids.device)
            full_ids = torch.cat([pad_ids, token_ids], dim=1)
        else:
            full_ids = token_ids

        S_full = x.shape[1]

        # 3. Precompute PLE signals for all layers
        if self.ple is not None:
            ple_signals = self.ple.get_signals(full_ids)  # list of (B, S, d_model)
        else:
            ple_signals = [torch.zeros_like(x)] * self.config.n_layers

        # 4. Decoder layers with alternating attention + shared KV
        shared_start = self.config.n_layers - self.config.num_kv_shared_layers
        last_local_kv  = None
        last_global_kv = None

        for i, layer in enumerate(self.layers):
            is_local = self.config.is_local_layer[i]
            is_shared = (i >= shared_start)

            # Choose RoPE
            rope = self.rope_local if is_local else self.rope_global

            # Shared KV: reuse last computed K,V for this attention type
            cached_kv = None
            if is_shared:
                cached_kv = last_local_kv if is_local else last_global_kv

            # Forward through layer
            x, kv = layer(x, rope, ple_signals[i], cached_kv=cached_kv)

            # Store K,V for potential sharing
            if not is_shared:
                if is_local:
                    last_local_kv = kv
                else:
                    last_global_kv = kv

        # 5. Final norm + output projection
        x = self.norm(x)
        logits = self.head(x)  # (B, S, vocab_size)
        return logits


# Instantiate
model = Gemma4E2B(config).to(device)

total_params  = sum(p.numel() for p in model.parameters())
embed_params  = sum(p.numel() for p in model.token_embed.parameters())
ple_params    = sum(p.numel() for p in model.ple.parameters()) if model.ple else 0
layer_params  = sum(p.numel() for p in model.layers.parameters())

print("=" * 55)
print("  Gemma 4 E2B Demo Model")
print("=" * 55)
print(f"  Token embedding:   {embed_params:>10,}")
print(f"  Per-Layer Embeds:  {ple_params:>10,}")
print(f"  Decoder layers:    {layer_params:>10,}")
print(f"  Total params:      {total_params:>10,}")
print(f"  (Real E2B: ~5.1B with embeddings, 2.3B effective)")
print("=" * 55)

---

### Part 10: Forward Pass & Inference Demo

🎯 **What it does**: Run a full text + image forward pass through the E2B model

🔧 **Why it matters**: Trace how tokens and image embeddings flow through PLE, alternating attention, and shared KV — seeing all innovations work together.

In [ ]:
model.eval()

# ── Text-only forward pass ─────────────────────────────────
print("Test 1: Text-only forward pass")
B, S = 2, 48
token_ids = torch.randint(0, config.vocab_size, (B, S)).to(device)

with torch.no_grad():
    logits = model(token_ids)

print(f"  Input tokens:    {token_ids.shape}")
print(f"  Output logits:   {logits.shape}")
print(f"  Next-token pred: argmax over {config.vocab_size} vocab")
print()

# ── Multimodal forward pass ────────────────────────────────
print("Test 2: Multimodal (image + text) forward pass")
vision_encoder = VisionPatchEmbedding(config).to(device)

# Simulate an image input
dummy_img = torch.randn(B, 3, 224, 224).to(device)
img_tokens = vision_encoder(dummy_img, token_budget=140)   # 140 visual tokens

# Encode with model (image prepended to text)
text_ids = torch.randint(0, config.vocab_size, (B, 20)).to(device)  # 20 text tokens

with torch.no_grad():
    mm_logits = model(text_ids, image_tokens=img_tokens)

print(f"  Image tokens:    {img_tokens.shape}  (140 visual tokens)")
print(f"  Text tokens:     {text_ids.shape}")
print(f"  Total sequence:  {140 + 20} tokens")
print(f"  Output logits:   {mm_logits.shape}")
print()

# ── Greedy decode demo ─────────────────────────────────────
print("Test 3: Simple greedy decoding (5 steps)")
prompt = torch.randint(0, config.vocab_size, (1, 10)).to(device)  # 10-token prompt
generated = prompt.clone()

with torch.no_grad():
    for step in range(5):
        logits_step = model(generated)
        next_token  = logits_step[:, -1, :].argmax(dim=-1, keepdim=True)
        generated   = torch.cat([generated, next_token], dim=1)
        print(f"  Step {step+1}: generated token #{next_token.item():4d} "
              f"(sequence length: {generated.shape[1]})")

print(f"\n  Final sequence length: {generated.shape[1]} tokens")

In [ ]:
# ── Architecture summary visualization ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Gemma 4 vs. predecessors feature table
ax = axes[0]
ax.axis('off')

features = [
    ('Feature', 'Gemma 3', 'Gemma 4 E2B', 'Importance'),
    ('Attention', 'Global', 'Local+Global', 'Efficiency ↑'),
    ('Position enc.', 'RoPE', 'Dual RoPE', 'Long ctx ↑'),
    ('KV efficiency', 'Standard', 'Shared KV', 'Memory ↓'),
    ('Layer cond.', 'None', 'PLE ✨', 'Spec. ↑'),
    ('Vision', 'Fixed tokens', 'Variable budget', 'Flex ↑'),
    ('Modalities', 'Text+Image', '+Audio+Video', 'Rich ↑'),
    ('Context', '128K', '128K (local+global)', 'Same ✓'),
    ('Norm', 'RMSNorm', 'RMSNorm', 'Same ✓'),
    ('FFN', 'SwiGLU', 'SwiGLU', 'Same ✓'),
]

col_widths = [0.28, 0.22, 0.28, 0.22]
col_x = [0.0, 0.28, 0.50, 0.78]
row_h = 1.0 / len(features)

for r, row in enumerate(features):
    y = 1.0 - (r + 0.5) * row_h
    is_header = (r == 0)

    for c, (val, x) in enumerate(zip(row, col_x)):
        bg = '#2c3e50' if is_header else ('#eaf4fb' if r % 2 == 0 else 'white')
        if not is_header and '✨' in val:
            bg = '#fef9e7'
        rect = mpatches.FancyBboxPatch((x, y - row_h/2 + 0.005),
                                        col_widths[c] - 0.01, row_h - 0.01,
                                        boxstyle='round,pad=0.005',
                                        facecolor=bg, transform=ax.transAxes,
                                        clip_on=False, linewidth=0)
        ax.add_patch(rect)
        color = 'white' if is_header else '#2c3e50'
        weight = 'bold' if is_header else 'normal'
        ax.text(x + col_widths[c]/2, y, val, ha='center', va='center',
                fontsize=8, color=color, fontweight=weight,
                transform=ax.transAxes)

ax.set_title('Gemma 4 E2B vs. Gemma 3: Key Differences', fontweight='bold', fontsize=11)

# Right: Parameter breakdown
categories = ['Token\nEmbedding', 'PLE\n(novel)', 'Attention\nLayers', 'FFN\nLayers', 'Norm\n+ Head']
sizes = [
    config.vocab_size * config.d_model,
    config.vocab_size * config.ple_dim + config.n_layers * (config.ple_dim * config.d_model + config.d_model**2),
    config.n_layers * (config.n_heads * config.head_dim * config.d_model + 
                       2 * config.n_kv_heads * config.head_dim * config.d_model +
                       config.n_heads * config.head_dim * config.d_model),
    config.n_layers * (config.d_model * config.d_ff * 3),
    config.n_layers * 2 * config.d_model + config.d_model,
]
colors_pie = ['#3498db', '#f39c12', '#e74c3c', '#2ecc71', '#9b59b6']
explode = [0, 0.08, 0, 0, 0]  # explode PLE slice

axes[1].pie(sizes, labels=categories, colors=colors_pie, explode=explode,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
axes[1].set_title('Demo Model: Parameter Distribution\n(PLE exploded = novel component)',
                   fontweight='bold', fontsize=11)

plt.suptitle('Gemma 4 E2B Architecture Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

### Summary

#### What We Built

A complete **Gemma 4 E2B-style decoder** from scratch, implementing all five key innovations:

| Innovation | What It Does | Benefit |
|-----------|-------------|--------|
| **Alternating Local/Global Attention** | Even layers: 512-token window; odd layers: full causal | O(n·w) + long-range context |
| **Dual RoPE** | θ=10K for local, θ=1M for global | Position precision at 128K context |
| **Per-Layer Embeddings (PLE)** | Small embedding table injected into every layer | Layer-specific token specialization |
| **Shared KV Cache** | Last N layers reuse K,V from previous | Memory and compute reduction |
| **Variable Token Budget** | Vision encoder outputs 70–1120 tokens | Speed/quality tradeoff control |

#### Gemma 4 E2B vs. Prior Art

```
Standard LLM:       embed → [global attn → FFN] × N → head

LLaMA-3:            embed → [GQA + RoPE → SwiGLU] × N → head

Gemma 4 E2B:        embed
                  + PLE ↓  (new!)
                    → [GQA(local|global) + dualRoPE → SwiGLU + PLE] × N
                    → [last N: reuse K,V]  (sharedKV new!)
                    → head
```

#### Real-World Gemma 4 E2B Performance

| Benchmark | E2B Score | Context |
|-----------|-----------|--------|
| MMLU Pro | 60.0% | Academic reasoning |
| GPQA Diamond | 43.4% | Expert-level QA |
| MMMU Pro (Vision) | 44.2% | Multimodal understanding |
| LiveCodeBench | 44.0% | Code generation |
| AIME 2026 | 37.5% | Competition math |

All at **2.3B effective parameters** — remarkable for an on-device model.

#### Use the Real Model

```python
from transformers import pipeline

pipe = pipeline("any-to-any", model="google/gemma-4-E2B-it")

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": "path/to/image.jpg"},
        {"type": "text",  "text": "What's in this image?"},
    ],
}]

response = pipe(messages, max_new_tokens=200)
```

---

🎉 **You've reverse-engineered Gemma 4 E2B!** The smallest model in Google's most capable open family, running on-device, understanding text + image + audio + video. Now you know exactly what's under the hood. 🔥

---

### 🗿 Chapter 8 Complete: The Smallest Giant

They said small models can't do multimodal.  
They said on-device AI means compromising quality.  
They said you need 27B parameters to understand images properly.

**Gemma 4 E2B said: hold my 2.3 billion parameters.**

---

#### 🌟 The Full Cave Map

| Chamber | Architecture | Core Insight |
|---------|-------------|--------------|
| **1** | PyTorch | Tensors are everything |
| **2** | Transformer | Attention is all you need |
| **3** | LLaMA | Modern LLMs improve the basics |
| **4** | ViT | Vision is just sequences |
| **5** | I-JEPA | Predict representations, not pixels |
| **6** | MoE | Not every expert needs every input |
| **7** | KD + Gemma 2 | A great teacher makes small models mighty |
| **8** | Gemma 4 E2B | **Small + smart + multimodal = the future** |

---

#### 🔑 The Five Keys You Now Hold

1. **Alternating attention** — don't choose between local and global. Take both.
2. **Dual RoPE** — the right positional encoding for the right range.
3. **Per-Layer Embeddings** — give each layer its own identity signal. Specialization is free.
4. **Shared KV Cache** — not every layer needs its own keys and values.
5. **Variable token budgets** — let the user decide the speed/quality tradeoff.

None of these ideas are magic. Each is a clean engineering decision backed by ablation studies.

That's what separates **good models** from **great models**.

---

> *"It's not the size of the model that matters. It's the quality of the ideas inside it."*

---

📖 *[Gemma 4 Blog — Hugging Face](https://huggingface.co/blog/gemma4)*  
🤗 *[google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it)*

---

🔥 **The cave is deep. Keep descending.** 🔥